# 02 — PM to next failure analysis

This notebook answers the core question:

> After a preventive maintenance event is completed, how long does the asset remain failure-free?

The notebook rebuilds the PM-to-next-failure linkage from the raw PM and failure tables and then compares on-time versus delayed PM.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path('..')
DATA = BASE / 'data' / 'processed'
OUT = BASE / 'outputs'


In [ ]:
pm = pd.read_csv(DATA / 'pm_events.csv', parse_dates=['pm_date'])
fail = pd.read_csv(DATA / 'failure_events.csv', parse_dates=['failure_date'])

In [ ]:
pm = pm.sort_values(['asset_id', 'pm_date']).copy()
fail = fail.sort_values(['asset_id', 'failure_date']).copy()

linked_rows = []
for asset_id, gpm in pm.groupby('asset_id'):
    gf = fail[fail['asset_id'] == asset_id]
    fdates = gf['failure_date'].to_numpy()
    fids = gf['failure_event_id'].to_numpy()
    fcosts = gf['total_cost'].to_numpy() if len(gf) else np.array([])
    for _, row in gpm.iterrows():
        idx = np.searchsorted(fdates, np.datetime64(row['pm_date']), side='right') if len(fdates) else 0
        if idx < len(fdates):
            next_date = pd.Timestamp(fdates[idx])
            linked_rows.append({
                'pm_event_id': row['pm_event_id'],
                'asset_id': asset_id,
                'pm_date': row['pm_date'],
                'delay_days': row['delay_days'],
                'ontime_flag': row['ontime_flag'],
                'next_failure_event_id': fids[idx],
                'next_failure_date': next_date,
                'days_to_next_failure': (next_date - row['pm_date']).days,
                'next_failure_cost': fcosts[idx],
            })
        else:
            linked_rows.append({
                'pm_event_id': row['pm_event_id'],
                'asset_id': asset_id,
                'pm_date': row['pm_date'],
                'delay_days': row['delay_days'],
                'ontime_flag': row['ontime_flag'],
                'next_failure_event_id': pd.NA,
                'next_failure_date': pd.NaT,
                'days_to_next_failure': pd.NA,
                'next_failure_cost': pd.NA,
            })

linked = pd.DataFrame(linked_rows)
linked.head()

In [ ]:
linked['days_to_next_failure'].dropna().describe()

## Compare on-time PM versus delayed PM

A good demo usually needs a clean comparison view. The chart below makes the story very clear for business audiences.

In [ ]:
plot_df = linked.dropna(subset=['days_to_next_failure']).copy()
plot_df['bucket'] = np.where(plot_df['ontime_flag'] == 1, 'On-time PM', 'Delayed PM')

summary = plot_df.groupby('bucket')['days_to_next_failure'].agg(['mean', 'median', 'count'])
summary

In [ ]:
plt.figure(figsize=(8, 5))
for bucket, group in plot_df.groupby('bucket'):
    plt.hist(group['days_to_next_failure'], bins=30, alpha=0.5, label=bucket)
plt.title('Days to next failure after PM')
plt.xlabel('Days')
plt.ylabel('PM events')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
bar_df = plot_df.groupby('bucket')['days_to_next_failure'].mean().reset_index()

plt.figure(figsize=(6, 4))
plt.bar(bar_df['bucket'], bar_df['days_to_next_failure'])
plt.title('Average failure-free days by PM timeliness')
plt.ylabel('Average days')
plt.tight_layout()
plt.show()

In [ ]:
thresholds = np.arange(0, 181, 10)
surv_ontime = [(plot_df.loc[plot_df['ontime_flag'] == 1, 'days_to_next_failure'] > t).mean() for t in thresholds]
surv_delayed = [(plot_df.loc[plot_df['ontime_flag'] == 0, 'days_to_next_failure'] > t).mean() for t in thresholds]

plt.figure(figsize=(8, 5))
plt.plot(thresholds, surv_ontime, label='On-time PM')
plt.plot(thresholds, surv_delayed, label='Delayed PM')
plt.title('Failure-free probability after PM')
plt.xlabel('Days after PM')
plt.ylabel('Probability of no failure yet')
plt.legend()
plt.tight_layout()
plt.show()